## 01. Create Discovery Golden Dataset

### Why are we implementing this?

We need a separate golden dataset to evaluate whether SEVA_AI can identify relevant government schemes from a citizen's real-world problem.

Unlike our existing FAQ dataset, citizen queries must not contain scheme names.

Each query will have manually verified relevance judgments for multiple schemes.

We will use this dataset to evaluate:
- Scheme discovery accuracy
- Recall@K and Precision@K
- MRR and MAP
- nDCG with graded relevance

We will initially target 150 citizen queries.

In [2]:
import json
from pathlib import Path

PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

DISCOVERY_DATASET_DIR = (
    PROJECT_ROOT
    / "notebooks"
    / "golden_dataset"
    / "discovery"
)

DISCOVERY_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_PATH = (
    DISCOVERY_DATASET_DIR
    / "discovery_golden_dataset.jsonl"
)

SCHEMA_PATH = (
    DISCOVERY_DATASET_DIR
    / "discovery_schema.json"
)

schema = {
    "query_id": "Unique query identifier",
    "citizen_query": "Citizen's actual problem without scheme names",
    "category": "Government scheme category",
    "answerable": "Boolean",
    "relevant_scheme_ids": [
        "Verified relevant scheme IDs"
    ],
    "relevance_judgments": [
        {
            "scheme_id": "Government scheme ID",
            "relevance_grade": 2,
            "rationale": "Why this scheme is relevant",
            "source_url": "Official verification URL"
        }
    ]
}

with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4, ensure_ascii=False)

if not DATASET_PATH.exists():
    DATASET_PATH.touch()

print("Discovery benchmark initialized!")
print("Dataset:", DATASET_PATH)
print("Schema:", SCHEMA_PATH)

Discovery benchmark initialized!
Dataset: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_golden_dataset.jsonl
Schema: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_schema.json


## 02. Analyze Scheme Categories

### Why are we implementing this?

We need to understand the distribution of government schemes before creating our 150 citizen queries.

We will inspect the existing scheme dataset to identify:
- Available scheme categories
- Number of schemes per category
- Available metadata fields
- Sample scheme records

This analysis will help us create a balanced discovery benchmark across different government domains.

In [3]:
import json
from collections import Counter
from pathlib import Path

SCHEME_PATH ="G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\data\\analysis\\all_schemes.json"

with open(SCHEME_PATH, "r", encoding="utf-8") as f:
    scheme_data = json.load(f)

if isinstance(scheme_data, list):
    schemes = scheme_data
elif isinstance(scheme_data, dict):
    schemes = scheme_data.get("schemes", [])
else:
    raise ValueError("Unexpected scheme dataset structure")

print("Total schemes:", len(schemes))

if not schemes:
    raise ValueError("No scheme records found.")

print("\nAvailable fields:")
print(list(schemes[0].keys()))

category_counts = Counter()

for scheme in schemes:
    categories = scheme.get("categories", [])

    if isinstance(categories, str):
        categories = [categories]

    category_counts.update(
        category for category in categories if category
    )

print("\nCategory distribution:")

for category, count in category_counts.most_common():
    print(f"{category}: {count}")

print("\nSample scheme:")
print(json.dumps(schemes[0], indent=2, ensure_ascii=False))

Total schemes: 4428

Available fields:
['scheme_id', 'internal_id', 'slug', 'name', 'short_title', 'level', 'ministry', 'nodal_department', 'department', 'states', 'categories', 'tags', 'target_beneficiaries', 'status', 'open_date', 'close_date', 'brief_description', 'details', 'benefits', 'exclusions', 'eligibility', 'application_process', 'documents_required', 'faqs', 'apply_url', 'apply_links', 'official_links', 'myscheme_url', 'source', 'fetched_on']

Category distribution:
Social welfare & Empowerment: 1331
Education & Learning: 1023
Agriculture,Rural & Environment: 808
Business & Entrepreneurship: 730
Women and Child: 426
Skills & Employment: 368
Banking,Financial Services and Insurance: 308
Health & Wellness: 263
Sports & Culture: 261
Housing & Shelter: 126
Science, IT & Communications: 114
Transport & Infrastructure: 101
Travel & Tourism: 95
Utility & Sanitation: 56
Public Safety,Law & Justice: 31

Sample scheme:
{
  "scheme_id": "LBezF5gB5uF80_MGGY81",
  "internal_id": "628727

## Full Scheme Dataset Validation

Validate the complete government scheme dataset before building the citizen-discovery benchmark. This checks data integrity, category coverage, scheme status, and geographic distribution.

In [4]:
import json
from pathlib import Path
from collections import Counter

SCHEME_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\notebooks\data\analysis\all_schemes.json")

with open(SCHEME_PATH, "r", encoding="utf-8") as file:
    schemes = json.load(file)

print("Total schemes:", len(schemes))
print("Unique scheme IDs:", len({s.get("scheme_id") for s in schemes}))
print("Categories:", len({
    category
    for scheme in schemes
    for category in scheme.get("categories", [])
}))

category_counts = Counter(
    category
    for scheme in schemes
    for category in scheme.get("categories", [])
)

status_counts = Counter(
    scheme.get("status", "unknown")
    for scheme in schemes
)

level_counts = Counter(
    scheme.get("level", "unknown")
    for scheme in schemes
)

state_counts = Counter(
    state
    for scheme in schemes
    for state in scheme.get("states", [])
)

print("\nCategory distribution:")
for category, count in category_counts.most_common():
    print(f"{category}: {count}")

print("\nStatus distribution:", status_counts)
print("\nLevel distribution:", level_counts)
print("\nTop 15 states:", state_counts.most_common(15))

Total schemes: 4428
Unique scheme IDs: 4428
Categories: 15

Category distribution:
Social welfare & Empowerment: 1331
Education & Learning: 1023
Agriculture,Rural & Environment: 808
Business & Entrepreneurship: 730
Women and Child: 426
Skills & Employment: 368
Banking,Financial Services and Insurance: 308
Health & Wellness: 263
Sports & Culture: 261
Housing & Shelter: 126
Science, IT & Communications: 114
Transport & Infrastructure: 101
Travel & Tourism: 95
Utility & Sanitation: 56
Public Safety,Law & Justice: 31

Status distribution: Counter({'active': 4384, 'closed': 44})

Level distribution: Counter({'State': 3735, 'Central': 693})

Top 15 states: [('All', 673), ('Gujarat', 594), ('Uttarakhand', 416), ('Madhya Pradesh', 263), ('Goa', 247), ('Puducherry', 245), ('Haryana', 224), ('Tamil Nadu', 212), ('Rajasthan', 147), ('West Bengal', 105), ('Chhattisgarh', 103), ('Bihar', 102), ('Assam', 86), ('Jharkhand', 86), ('Meghalaya', 85)]


## Geographic Coverage Analysis

Analyze scheme availability by state and scheme level. This helps us design citizen queries that test both Central and State government scheme discovery, including location-specific eligibility.

In [5]:
from collections import Counter

central_schemes = [
    s for s in schemes if s.get("level") == "Central"
]

state_schemes = [
    s for s in schemes if s.get("level") == "State"
]

state_distribution = Counter(
    state
    for scheme in state_schemes
    for state in scheme.get("states", [])
    if state != "All"
)

print("Central schemes:", len(central_schemes))
print("State schemes:", len(state_schemes))

print("\nState-wise scheme distribution:")
for state, count in state_distribution.most_common():
    print(f"{state}: {count}")

print("\nCentral scheme state values:")
print(Counter(
    state
    for scheme in central_schemes
    for state in scheme.get("states", [])
))

Central schemes: 693
State schemes: 3735

State-wise scheme distribution:
Gujarat: 592
Uttarakhand: 415
Madhya Pradesh: 261
Goa: 246
Puducherry: 245
Haryana: 224
Tamil Nadu: 206
Rajasthan: 145
West Bengal: 104
Chhattisgarh: 101
Bihar: 101
Jharkhand: 85
Meghalaya: 79
Kerala: 78
Assam: 77
Maharashtra: 77
Odisha: 77
Himachal Pradesh: 73
Dadra & Nagar Haveli and Daman & Diu: 59
Karnataka: 54
Delhi: 49
Andhra Pradesh: 44
Punjab: 41
Uttar Pradesh: 41
Jammu and Kashmir: 39
Tripura: 36
Arunachal Pradesh: 35
Manipur: 24
Sikkim: 21
Mizoram: 21
Telangana: 21
Chandigarh: 19
Andaman and Nicobar Islands: 17
Nagaland: 16
Lakshadweep: 9
Ladakh: 3

Central scheme state values:
Counter({'All': 673, 'Assam': 9, 'Tripura': 7, 'Arunachal Pradesh': 7, 'Manipur': 7, 'Mizoram': 7, 'Tamil Nadu': 6, 'Sikkim': 6, 'Meghalaya': 6, 'Kerala': 6, 'Nagaland': 6, 'Karnataka': 4, 'Jammu and Kashmir': 3, 'Gujarat': 2, 'Rajasthan': 2, 'Madhya Pradesh': 2, 'Chhattisgarh': 2, 'Odisha': 2, 'Maharashtra': 1, 'West Bengal': 1,

## National Geographic Coverage Inventory

Prepare a complete state and union territory coverage inventory from the 4,428-scheme dataset. This will help distribute the citizen-discovery benchmark across India without prioritizing one state.

In [6]:
from collections import Counter

state_scheme_counts = Counter(
    state
    for scheme in schemes
    if scheme.get("level") == "State"
    for state in scheme.get("states", [])
    if state != "All"
)

print("Total states and UTs:", len(state_scheme_counts))
print("\nNational geographic coverage:\n")

for index, (state, count) in enumerate(
    sorted(state_scheme_counts.items()), start=1
):
    print(f"{index:02d}. {state}: {count} schemes")

Total states and UTs: 36

National geographic coverage:

01. Andaman and Nicobar Islands: 17 schemes
02. Andhra Pradesh: 44 schemes
03. Arunachal Pradesh: 35 schemes
04. Assam: 77 schemes
05. Bihar: 101 schemes
06. Chandigarh: 19 schemes
07. Chhattisgarh: 101 schemes
08. Dadra & Nagar Haveli and Daman & Diu: 59 schemes
09. Delhi: 49 schemes
10. Goa: 246 schemes
11. Gujarat: 592 schemes
12. Haryana: 224 schemes
13. Himachal Pradesh: 73 schemes
14. Jammu and Kashmir: 39 schemes
15. Jharkhand: 85 schemes
16. Karnataka: 54 schemes
17. Kerala: 78 schemes
18. Ladakh: 3 schemes
19. Lakshadweep: 9 schemes
20. Madhya Pradesh: 261 schemes
21. Maharashtra: 77 schemes
22. Manipur: 24 schemes
23. Meghalaya: 79 schemes
24. Mizoram: 21 schemes
25. Nagaland: 16 schemes
26. Odisha: 77 schemes
27. Puducherry: 245 schemes
28. Punjab: 41 schemes
29. Rajasthan: 145 schemes
30. Sikkim: 21 schemes
31. Tamil Nadu: 206 schemes
32. Telangana: 21 schemes
33. Tripura: 36 schemes
34. Uttar Pradesh: 41 schemes
35. 

## National Discovery Benchmark Allocation

Create a balanced allocation of 150 citizen queries across all 15 scheme categories and 36 Indian States and Union Territories. Include both Central and State scheme scenarios while ensuring every state and UT is represented.

One important distinction: 10% is our sampling target, not necessarily one unique question per scheme. We should generate questions from scheme information, then verify that each question has the correct relevant scheme IDs. Some citizen questions may legitimately match multiple schemes.

In [7]:
import math
import pandas as pd

state_targets = pd.DataFrame([
    {
        "state": state,
        "scheme_count": count,
        "target_questions": math.ceil(count * 0.10)
    }
    for state, count in state_scheme_counts.items()
])

central_count = len(central_schemes)
central_target = math.ceil(central_count * 0.10)

print(state_targets.to_string(index=False))
print("\nState questions:", state_targets["target_questions"].sum())
print("Central questions:", central_target)
print(
    "Total target questions:",
    state_targets["target_questions"].sum() + central_target
)

                               state  scheme_count  target_questions
                               Delhi            49                 5
                              Sikkim            21                 3
                             Haryana           224                23
                           Karnataka            54                 6
                             Mizoram            21                 3
                    Himachal Pradesh            73                 8
                             Gujarat           592                60
                               Assam            77                 8
                           Jharkhand            85                 9
                            Nagaland            16                 2
                             Manipur            24                 3
                           Rajasthan           145                15
                         West Bengal           104                11
                         Maharasht

## Discovery Benchmark Scheme Sampling

Select approximately 10% of State schemes for each State/UT and 10% of Central schemes. Each selected scheme becomes a source for generating one natural-language citizen query. Preserve the original scheme information and geographic scope for relevance judgments.

In [10]:
import json
import math
import random
from pathlib import Path

random.seed(42)

discovery_dir = Path(
    r"G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery"
)
discovery_dir.mkdir(parents=True, exist_ok=True)

generation_tasks = []

# Sample State schemes separately for each State/UT
for state, count in state_scheme_counts.items():
    state_records = [
        scheme for scheme in schemes
        if scheme.get("level") == "State"
        and state in scheme.get("states", [])
    ]

    target = math.ceil(len(state_records) * 0.10)
    selected = random.sample(state_records, target)

    for scheme in selected:
        generation_tasks.append({
            "generation_id": f"STATE_{state}_{scheme['scheme_id']}",
            "scope": "State",
            "target_state": state,
            "scheme": scheme
        })

# Sample Central schemes
central_target = math.ceil(len(central_schemes) * 0.10)
selected_central = random.sample(
    central_schemes, central_target
)

for scheme in selected_central:
    generation_tasks.append({
        "generation_id": f"CENTRAL_{scheme['scheme_id']}",
        "scope": "Central",
        "target_state": "All India",
        "scheme": scheme
    })

output_path = discovery_dir / "discovery_generation_tasks.jsonl"

with open(output_path, "w", encoding="utf-8") as file:
    for task in generation_tasks:
        file.write(
            json.dumps(task, ensure_ascii=False) + "\n"
        )

print("Generation tasks:", len(generation_tasks))
print("State tasks:", sum(t["scope"] == "State" for t in generation_tasks))
print("Central tasks:", sum(t["scope"] == "Central" for t in generation_tasks))
print("Saved:", output_path)

Generation tasks: 462
State tasks: 392
Central tasks: 70
Saved: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_generation_tasks.jsonl


In [8]:
import math
import random

random.seed(42)

generation_tasks = []

# State scheme sampling
for state in state_scheme_counts:
    state_records = [
        scheme for scheme in schemes
        if scheme.get("level") == "State"
        and state in scheme.get("states", [])
    ]

    target = math.ceil(len(state_records) * 0.10)
    selected = random.sample(state_records, target)

    for scheme in selected:
        generation_tasks.append({
            "generation_id": f"STATE_{state}_{scheme['scheme_id']}",
            "scope": "State",
            "target_state": state,
            "scheme": scheme
        })

# Central scheme sampling
central_target = math.ceil(len(central_schemes) * 0.10)

for scheme in random.sample(central_schemes, central_target):
    generation_tasks.append({
        "generation_id": f"CENTRAL_{scheme['scheme_id']}",
        "scope": "Central",
        "target_state": "All India",
        "scheme": scheme
    })

print("Total generation tasks:", len(generation_tasks))

Total generation tasks: 462


In [10]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
import os

# google_api = os.getenv("GOOGLE_API_KEY")
groq_api = os.getenv("GROQ_API_KEY")

if groq_api is None:
    raise ValueError("GROQ_API_KEY is not set in the environment variables")

llm = ChatGroq(
    api_key=groq_api,
    model="openai/gpt-oss-120b"
)

## Scheme Discovery Golden Dataset Schema

Define the standard structure for evaluating citizen-to-scheme discovery.

Each record represents one citizen problem and contains:
- Optional citizen profile filters.
- Natural-language citizen query.
- Multiple relevant scheme IDs.
- Graded relevance judgments for each evaluated scheme.
- Evidence explaining why each scheme is relevant.

This dataset will evaluate scheme discovery, not scheme-specific question answering.

In [41]:
import json
from pathlib import Path

discovery_dir = Path(
    r"G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery"
)

discovery_dir.mkdir(parents=True, exist_ok=True)

discovery_schema = {
    "query_id": "Unique citizen query ID",

    "user_profile": {
        "state": "Indian state or union territory",
        "caste_category": "Optional caste category",
        "community_category": "Optional community category",
        "beneficiary_group": "Optional beneficiary group",
        "age": "Optional age",
        "annual_family_income": "Optional annual family income",
        "scheme_category": "Optional primary scheme category"
    },

    "citizen_query": "Natural-language description of citizen's problem",

    "relevant_scheme_ids": [
        "All verified relevant scheme IDs"
    ],

    "relevance_judgments": [
        {
            "scheme_id": "Government scheme ID",
            "relevance_grade": "0, 1, or 2",
            "rationale": "Why this scheme is relevant",
            "evidence_fields": [
                "Original scheme fields supporting relevance"
            ]
        }
    ]
}

schema_path = discovery_dir / "discovery_schema.json"
dataset_path = discovery_dir / "discovery_golden_dataset.jsonl"

with open(schema_path, "w", encoding="utf-8") as file:
    json.dump(
        discovery_schema,
        file,
        ensure_ascii=False,
        indent=2
    )

if not dataset_path.exists():
    dataset_path.touch()

print("Schema created:", schema_path)
print("Dataset initialized:", dataset_path)

Schema created: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_schema.json
Dataset initialized: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_golden_dataset.jsonl


### Pilot Citizen Query Generation

Generate one realistic citizen problem using a sampled government scheme as the
source of inspiration.

The citizen query must describe the problem or need in natural language.
Do not mention the scheme name, scheme ID, or repeat profile filters such as
state, age, caste, or income in the query.

This is only a candidate query. Its relevant schemes will be identified and
verified separately against the complete scheme dataset.

In [15]:
import json

task = generation_tasks[0]
scheme = task["scheme"]

scheme_context = {
    "name": scheme.get("name"),
    "level": scheme.get("level"),
    "states": scheme.get("states"),
    "categories": scheme.get("categories"),
    "target_beneficiaries": scheme.get("target_beneficiaries"),
    "brief_description": scheme.get("brief_description"),
    "benefits": scheme.get("benefits"),
    "eligibility": scheme.get("eligibility"),
}

prompt = f"""
You are creating a golden dataset for an Indian government scheme discovery system.

Generate ONE realistic citizen problem query based on the scheme information below.

SCHEME INFORMATION:
{json.dumps(scheme_context, ensure_ascii=False, indent=2)}

STRICT RULES:
1. Generate one short citizen problem query (8–20 words).
2. Use simple, natural, conversational language.
3. Describe only the citizen's primary need or problem.
4. Do not mention scheme names, IDs, departments, or government scheme descriptions.
5. Do not invent personal details such as age, income, caste, or occupation.
6. Do not repeat profile filters in the query.
7. Avoid unnecessary background, explanations, and multiple questions.
8. Do not ask about eligibility, documents, or application procedures.

Return only valid JSON:
{{
    "citizen_query": "..."
}}
"""

response = llm.invoke(prompt)

result = response.content
if isinstance(result, list):
    result = "".join(
        item.get("text", "") if isinstance(item, dict) else str(item)
        for item in result
    )

result = result.strip().removeprefix("```json").removesuffix("```").strip()
pilot_query = json.loads(result)

print("Source scheme:", scheme.get("name"))
print("Target state:", task.get("target_state"))
print("Generated citizen query:", pilot_query["citizen_query"])

Source scheme: Transport Loan Scheme
Target state: Delhi
Generated citizen query: How can I get a loan to buy a small commercial vehicle?


### Generate Discovery Query Candidates

Generate one short citizen-problem query for each sampled scheme.

Keep the citizen query independent of profile filters. Store the target state
separately as structured profile information.

These queries are candidates only. Relevant scheme IDs must be identified
and validated against the complete scheme dataset before adding them to the
discovery golden dataset.

In [16]:
output_path = discovery_dir / "discovery_query_candidates.jsonl"

def parse_llm_json(response):
    content = response.content

    if isinstance(content, list):
        content = "".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in content
        )

    content = content.strip()

    if content.startswith("```"):
        content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    return json.loads(content)


candidates = []

for index, task in enumerate(generation_tasks[:10], start=1):
    scheme = task["scheme"]

    scheme_context = {
        "name": scheme.get("name"),
        "level": scheme.get("level"),
        "categories": scheme.get("categories"),
        "target_beneficiaries": scheme.get("target_beneficiaries"),
        "brief_description": scheme.get("brief_description"),
        "benefits": scheme.get("benefits"),
    }

    prompt = f"""
Generate ONE realistic citizen problem query for an Indian government
scheme discovery benchmark.

Source scheme information:
{json.dumps(scheme_context, ensure_ascii=False)}

Rules:
- Use 8–20 words.
- Use simple, natural language.
- Describe only the citizen's primary need.
- Do not mention the scheme name, ID, department, or scheme description.
- Do not invent age, income, caste, occupation, or other personal details.
- Do not mention the target state in the query.
- Do not ask about eligibility, documents, or application procedures.
- Avoid unnecessary background and multiple questions.

Return only valid JSON:
{{"citizen_query": "..."}}
"""

    response = llm.invoke(prompt)
    generated = parse_llm_json(response)

    candidates.append({
        "query_id": f"DISC_{index:04d}",
        "citizen_query": generated["citizen_query"],
        "user_profile": {
            "state": task.get("target_state")
        },
        "seed_scheme_id": scheme.get("scheme_id"),
        "seed_scheme_name": scheme.get("name"),
        "scope": task.get("scope"),
    })

with open(output_path, "w", encoding="utf-8") as file:
    for candidate in candidates:
        file.write(json.dumps(candidate, ensure_ascii=False) + "\n")

print(f"Generated candidates: {len(candidates)}")
print(f"Saved to: {output_path}")

for candidate in candidates:
    print(f"\n{candidate['query_id']}: {candidate['citizen_query']}")

Generated candidates: 10
Saved to: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_query_candidates.jsonl

DISC_0001: I need a loan to buy a light motor vehicle for my business

DISC_0002: How can I get funds to organize a Dr. Ambedkar birth anniversary event?

DISC_0003: I need financial support for my cancer treatment in a government hospital.

DISC_0004: I require free travel, food, and lodging for a pilgrimage because I cannot manage it alone.

DISC_0005: I need a guaranteed loan to fund my college education.

DISC_0006: I need financial support because I am an unmarried woman from a low-income family.

DISC_0007: Can I get cashless hospital admission for my family’s emergency surgery?

DISC_0008: I need financial assistance for my daughter's marriage.

DISC_0009: I need monthly financial help for my family's resettlement after moving from Kashmir.

DISC_0010: How can I receive reimbursement for my train fare to Varanasi pilgrimage?


### Inspect Scheme Categories

Identify the exact category names and scheme distribution from the original
scheme dataset. We will use these categories to organize our few-shot examples.

### Select Social Welfare Few-Shot Examples

Inspect original scheme records in the Social welfare & Empowerment category.
Identify diverse schemes that can demonstrate how original scheme information
is transformed into short, natural citizen problem queries.

In [22]:
import json

task = generation_tasks[0]
scheme = task["scheme"]

scheme_context = {
    "name": scheme.get("name"),
    "level": scheme.get("level"),
    "states": scheme.get("states"),
    "categories": scheme.get("categories"),
    "target_beneficiaries": scheme.get("target_beneficiaries"),
    "brief_description": scheme.get("brief_description"),
    "benefits": scheme.get("benefits"),
    "eligibility": scheme.get("eligibility"),
}

few_shot_examples = [
    {
        "input": {
            "scheme_context": {
                "scheme_id": "3ca0F5gB0AGc0F0iDzOE",
                "name": "Financial Assistance For The Marriage Of Daughters Of Poor Widows And Orphan Girls",
                "level": "State",
                "states": ["Delhi"],
                "categories": [
                    "Social welfare & Empowerment",
                    "Women and Child"
                ],
                "target_beneficiaries": [
                    {"value": "individual", "label": "Individual"}
                ],
                "brief_description": "This is a welfare scheme by the GNCTD to provide financial assistance to poor widows for performing the marriage of their daughters and to provide financial assistance to the guardians including Homes/Institutions or foster parents of an orphan girl or an orphan girl for her marriage.",
                "details": "The scheme provides financial assistance to poor widows for performing the marriage of their daughters (up to two daughters) and to guardians, institutions, foster parents or orphan girls for marriage.",
                "benefits": "One-time financial assistance of ₹30,000, up to two daughters only.",
                "eligibility": "Minimum five years residence in Delhi; annual income below ₹1,00,000; other specified conditions apply."
            }
        },
        "output": {
            "citizen_question": "I need financial help to arrange my daughter's marriage."
        }
    }
]

prompt = f"""
    Generate one realistic citizen problem question.

    Learn the input-output pattern from these examples:

    {json.dumps(few_shot_examples, ensure_ascii=False)}

    INPUT:
    {json.dumps(scheme_context, ensure_ascii=False)}

    Generate a question that represents the citizen's actual need.

    Rules:
    - Keep it short (8–20 words).
    - Use natural language.
    - Do not mention the scheme name.
    - Do not invent personal details.
    - Do not ask about eligibility or application procedures.

    OUTPUT:
    Return only JSON:
    {{"citizen_question": "..."}}
    """

### Test Few-Shot Citizen Query Generation

Use the manually curated marriage-assistance example as a few-shot example.
Pass the complete original Old Age Samman Allowance scheme context to the LLM.

Generate one short citizen problem question.

In [24]:
import json

# Retrieve actual scheme from original dataset
# current_scheme = next(
#     s for s in schemes
#     if s["scheme_id"] == "VaHgF5gBXu-iUKXq63iW"
# )

current_scheme = schemes[5]  # For demonstration, using the first scheme

scheme_context = {
    "scheme_id": current_scheme["scheme_id"],
    "name": current_scheme["name"],
    "level": current_scheme["level"],
    "states": current_scheme["states"],
    "categories": current_scheme["categories"],
    "target_beneficiaries": current_scheme["target_beneficiaries"],
    "brief_description": current_scheme["brief_description"],
    "details": current_scheme["details"],
    "benefits": current_scheme["benefits"],
    "eligibility": current_scheme["eligibility"]
}

prompt = f"""
Generate one realistic citizen problem question.

FEW-SHOT EXAMPLE:
{json.dumps(few_shot_examples[0], ensure_ascii=False)}

INPUT:
{json.dumps({"scheme_context": scheme_context}, ensure_ascii=False)}

Generate a citizen question that follows the example's input-output pattern.

Rules:
- Keep it short (8–20 words).
- Use natural language.
- Focus on the primary need addressed by the scheme.
- Do not mention the scheme name.
- Do not invent personal details.
- Do not ask about eligibility, documents or application procedures.

Return only JSON:
{{"citizen_question": "..."}}
"""

response = llm.invoke(prompt)

result = parse_llm_json(response)

print("Scheme:", current_scheme["name"])
print("Generated citizen question:", result["citizen_question"])

Scheme: Post Matric Scholarship Scheme For The Students Belonging To Scheduled Tribe For Studies In India
Generated citizen question: I need financial support to continue my college studies.


### Build Category-Wise Few-Shot Library FINAL

Select two representative original schemes from each category.
Preserve their original scheme context and prepare the input-output
structure for manually curated citizen questions.

In [25]:
import json
import random

fields = [
    "scheme_id", "name", "level", "states", "categories",
    "target_beneficiaries", "brief_description", "details",
    "benefits", "eligibility"
]

rng = random.Random(42)
selected_ids = set()
fewshot_candidates = {}

for category in category_counts:

    records = [
        s for s in schemes
        if category in (s.get("categories") or [])
        and s["scheme_id"] not in selected_ids
        and s.get("brief_description")
        and s.get("benefits")
    ]

    selected = rng.sample(records, min(2, len(records)))
    fewshot_candidates[category] = []

    for scheme in selected:
        selected_ids.add(scheme["scheme_id"])

        context = {field: scheme.get(field) for field in fields}

        fewshot_candidates[category].append({
            "input": {"scheme_context": context},
            "output": {"citizen_question": ""}
        })

output_path = discovery_dir / "discovery_fewshot_candidates.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(fewshot_candidates, file, ensure_ascii=False, indent=2)

print("Categories:", len(fewshot_candidates))
print("Selected examples:", sum(map(len, fewshot_candidates.values())))
print("Saved:", output_path)

Categories: 15
Selected examples: 30
Saved: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_fewshot_candidates.json


## Step 1: Validate Discovery Few-Shot Dataset

The few-shot dataset provides category-specific examples to guide citizen-question generation. Before using it in the LLM prompt, we validate its structure, category distribution, scheme contexts, and generated questions to catch missing or incomplete examples.

In [27]:
import json
from pathlib import Path
from collections import Counter

# PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")
# DISCOVERY_DIR = PROJECT_ROOT / "notebooks" / "golden_dataset" / "discovery"

FEWSHOT_PATH = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\discovery\\discovery_fewshot_candidates.json"

with open(FEWSHOT_PATH, "r", encoding="utf-8") as f:
    fewshot_data = json.load(f)

total_examples = 0
missing_context = []
missing_questions = []
category_counts = {}

required_context_fields = [
    "scheme_id",
    "name",
    "level",
    "states",
    "categories",
    "target_beneficiaries",
    "brief_description",
    "details",
    "benefits",
    "eligibility"
]

for category, examples in fewshot_data.items():
    category_counts[category] = len(examples)
    total_examples += len(examples)

    for index, example in enumerate(examples):
        context = example.get("input", {}).get("scheme_context", {})
        question = example.get("output", {}).get("citizen_question", "")

        missing_fields = [
            field for field in required_context_fields
            if field not in context or context[field] is None
        ]

        if missing_fields:
            missing_context.append({
                "category": category,
                "example": index + 1,
                "missing_fields": missing_fields
            })

        if not isinstance(question, str) or not question.strip():
            missing_questions.append({
                "category": category,
                "example": index + 1
            })

print("Total categories:", len(fewshot_data))
print("Total examples:", total_examples)
print("Category distribution:", category_counts)
print("Missing context:", len(missing_context))
print("Missing questions:", len(missing_questions))

if missing_context:
    print("\nIncomplete contexts:")
    print(missing_context)

if missing_questions:
    print("\nMissing citizen questions:")
    print(missing_questions)

assert len(fewshot_data) == 15, "Expected 15 categories"
assert total_examples == 30, "Expected 30 examples"
assert not missing_context, "Some scheme contexts are incomplete"
assert not missing_questions, "Some citizen questions are missing"

print("\nDataset structural validation passed.")

Total categories: 15
Total examples: 30
Category distribution: {'Business & Entrepreneurship': 2, 'Banking,Financial Services and Insurance': 2, 'Social welfare & Empowerment': 2, 'Sports & Culture': 2, 'Women and Child': 2, 'Education & Learning': 2, 'Agriculture,Rural & Environment': 2, 'Skills & Employment': 2, 'Health & Wellness': 2, 'Housing & Shelter': 2, 'Public Safety,Law & Justice': 2, 'Science, IT & Communications': 2, 'Transport & Infrastructure': 2, 'Travel & Tourism': 2, 'Utility & Sanitation': 2}
Missing context: 0
Missing questions: 0

Dataset structural validation passed.


## Semantic Few-Shot Selection

Use the existing embedding model to find the two most similar few-shot examples for a target scheme. Similarity is calculated using cosine similarity between the scheme contexts.

## Category-Based Few-Shot Selection

Select few-shot examples by matching the target scheme's categories against the available few-shot dataset. No embeddings or additional classification model are required.

In [35]:
def select_fewshot_examples(scheme, fewshot_data, max_examples=4):
    scheme_categories = scheme.get("categories", [])

    selected = []

    for category in scheme_categories:
        examples = fewshot_data.get(category, [])

        for example in examples:
            selected.append({
                "category": category,
                "scheme_context": example["input"]["scheme_context"],
                "citizen_question": example["output"]["citizen_question"]
            })

    return selected[:max_examples]


# Test
selected_examples = select_fewshot_examples(
    test_scheme,
    fewshot_data,
    max_examples=4
)

for example in selected_examples:
    print(example["category"])
    print(example["scheme_context"]["name"])
    print(example["citizen_question"])
    print("-" * 50)

Business & Entrepreneurship
25% Capital Investment Subsidy Scheme
I want to start a small business but need financial help to set it up.
--------------------------------------------------
Business & Entrepreneurship
Scheme for Development Maintenance and Construction of Roads
My small business handles road construction, and I need a loan to purchase equipment.
--------------------------------------------------
Banking,Financial Services and Insurance
Green Business Scheme
I want to start an eco-friendly business but need an affordable loan to fund it.
--------------------------------------------------
Banking,Financial Services and Insurance
Term Loan Scheme (GTKVN) 
I need a loan to start a small business or support my traditional occupation.
--------------------------------------------------


In [ ]:
import json

# Retrieve actual scheme from original dataset
# current_scheme = next(
#     s for s in schemes
#     if s["scheme_id"] == "VaHgF5gBXu-iUKXq63iW"
# )

current_scheme = schemes[5]  # For demonstration, using the first scheme

scheme_context = {
    "scheme_id": current_scheme["scheme_id"],
    "name": current_scheme["name"],
    "level": current_scheme["level"],
    "states": current_scheme["states"],
    "categories": current_scheme["categories"],
    "target_beneficiaries": current_scheme["target_beneficiaries"],
    "brief_description": current_scheme["brief_description"],
    "details": current_scheme["details"],
    "benefits": current_scheme["benefits"],
    "eligibility": current_scheme["eligibility"]
}

prompt = f"""
Generate one realistic citizen problem question.

FEW-SHOT EXAMPLE:
{json.dumps(few_shot_examples[0], ensure_ascii=False)}

INPUT:
{json.dumps({"scheme_context": scheme_context}, ensure_ascii=False)}

Generate a citizen question that follows the example's input-output pattern.

Rules:
- Keep it short (8–20 words).
- Use natural language.
- Focus on the primary need addressed by the scheme.
- Do not mention the scheme name.
- Do not invent personal details.
- Do not ask about eligibility, documents or application procedures.

Return only JSON:
{{"citizen_question": "..."}}
"""

response = llm.invoke(prompt)

result = parse_llm_json(response)

print("Scheme:", current_scheme["name"])
print("Generated citizen question:", result["citizen_question"])

In [38]:
import json
import time
from pathlib import Path

from tqdm.auto import tqdm
from langchain_core.messages import HumanMessage

PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

SCHEMES_PATH = (
    PROJECT_ROOT / "notebooks/data/analysis/all_schemes.json"
)

FEWSHOT_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "discovery_fewshot_candidates.json"
)

OUTPUT_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "generated_discovery_questions.jsonl"
)

MAX_EXAMPLES = 4
QUESTIONS_PER_SCHEME = 5

GOLDEN_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "discovery_golden_dataset.jsonl"
)

with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    golden_schemes = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Golden dataset schemes:", len(golden_schemes))

with open(FEWSHOT_PATH, "r", encoding="utf-8") as f:
    fewshot_data = json.load(f)

print("Total schemes:", len(schemes))
print("Few-shot categories:", len(fewshot_data))


def select_fewshot_examples(
    scheme,
    fewshot_data,
    max_examples=MAX_EXAMPLES
):
    scheme_categories = scheme.get("categories", [])

    selected = []

    for category in scheme_categories:
        examples = fewshot_data.get(category, [])

        for example in examples:
            selected.append({
                "category": category,
                "scheme_context": example["input"]["scheme_context"],
                "citizen_question": example["output"]["citizen_question"]
            })

    return selected[:max_examples]



def build_discovery_prompt(target_scheme, selected_examples):

    examples_text = "\n\n".join(
        [
            f"Scheme Context:\n"
            f"{json.dumps(example['scheme_context'], ensure_ascii=False)}\n"
            f"Citizen Question:\n"
            f"{example['citizen_question']}"
            for example in selected_examples
        ]
    )

    target_context = json.dumps(
        target_scheme,
        ensure_ascii=False
    )

    return f"""
You are an expert in understanding Indian government welfare schemes.

Your task is to generate realistic citizen queries based on the target scheme.

FEW-SHOT EXAMPLES:
{examples_text}

TARGET SCHEME:
{target_context}

INSTRUCTIONS:
1. Generate 5 different natural citizen queries.
2. Each query should describe a real citizen need addressed by the scheme.
3. Keep each query between 8 and 20 words.
4. Do not mention the scheme name.
5. Do not invent personal details, demographics, or circumstances.
6. Do not ask about eligibility, documents, or application procedures.
7. Avoid repeating state, caste, age, and income filters because these are collected separately.
8. Follow the style of the examples without copying their questions.
9. Return valid JSON only.

OUTPUT FORMAT:
{{
    "citizen_questions": [
        "Question 1",
        "Question 2",
        "Question 3",
        "Question 4",
        "Question 5"
    ]
}}
"""

Total schemes: 4428
Few-shot categories: 15


In [ ]:
PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

SCHEMES_PATH = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\discovery\\discovery_generation_tasks.jsonl"

with open(SCHEMES_PATH, "r", encoding="utf-8") as f:
    g_scheme_data = json.load(f)

DISCOVERY_FIELDS = [
    "scheme_id",
    "name",
    "level",
    "states",
    "categories",
    "tags",
    "target_beneficiaries",
    "brief_description",
    "details",
    "benefits"
]

discovery_schemes = [
    {
        field: record["scheme"].get(field)
        for field in DISCOVERY_FIELDS
    }
    for record in g_scheme_data
]

Total records: 462
First scheme: Transport Loan Scheme


In [47]:
discovery_schemes[10]

{'scheme_id': 'Bhe7F5gB5uF80_MGVZFX',
 'name': 'Financial Assistance for Artificial Limbs to Workers and their Dependents- Haryana Labour Welfare Board',
 'level': 'State',
 'states': ['Haryana'],
 'categories': ['Social welfare & Empowerment'],
 'tags': ['Worker',
  'Financial Assistance',
  'Artificial Limbs',
  'PwD',
  'Disability',
  'Labour'],
 'target_beneficiaries': [{'value': 'individual', 'label': 'Individual'},
  {'value': 'family', 'label': 'Family'}],
 'brief_description': 'Under this scheme, financial assistance is provided for artificial limbs upto rates fixed by Saket Hospital, Chandimandir(Panchkula). This assistance is available to workers employed in industrial &commercial units in Haryana and their dependents, who become disabled due to the loss of vital organs.',
 'details': 'The scheme “Financial Assistance for Artificial Limbs to Workers and their Dependents” is implemented by the Haryana Labour Welfare Board (HLWB), Labour Department, Government of Haryana. Unde

In [39]:
def generate_citizen_questions(
    target_scheme,
    selected_examples
):

    prompt = build_discovery_prompt(
        target_scheme,
        selected_examples
    )

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        content = "".join(
            item.get("text", "")
            for item in content
            if isinstance(item, dict)
        )

    content = content.strip()

    # Handle accidental Markdown JSON fences
    if content.startswith("```"):
        content = content.split("\n", 1)[1]
        content = content.rsplit("```", 1)[0]

    result = json.loads(content)

    questions = result["citizen_questions"]

    if len(questions) != QUESTIONS_PER_SCHEME:
        raise ValueError(
            f"Expected 5 questions, received {len(questions)}"
        )

    return questions

In [44]:
selected_examples = select_fewshot_examples(
    scheme=discovery_schemes[0],
    fewshot_data=fewshot_data,
    max_examples=4
)

print("Target scheme:", discovery_schemes[0]["name"])
print("Selected examples:", len(selected_examples))

for example in selected_examples:
    print("\nCategory:", example["category"])
    print("Citizen question:", example["citizen_question"])

Target scheme: Transport Loan Scheme
Selected examples: 4

Category: Transport & Infrastructure
Citizen question: I have an orthopedic disability and need a tax concession when purchasing a car.

Category: Transport & Infrastructure
Citizen question: I run a handicraft business and need support to improve my production facilities and technology.

Category: Business & Entrepreneurship
Citizen question: I want to start a small business but need financial help to set it up.

Category: Business & Entrepreneurship
Citizen question: My small business handles road construction, and I need a loan to purchase equipment.


In [45]:
selected_examples = select_fewshot_examples(
    discovery_schemes[0],
    fewshot_data,
    max_examples=4
)

prompt = build_discovery_prompt(
    discovery_schemes[0],
    selected_examples
)

print(prompt)


You are an expert in understanding Indian government welfare schemes.

Your task is to generate realistic citizen queries based on the target scheme.

FEW-SHOT EXAMPLES:
Scheme Context:
{"scheme_id": "Ihe3F5gB5uF80_MGNJAo", "name": "GST Exemption Certificate Scheme", "level": "Central", "states": ["All"], "categories": ["Transport & Infrastructure", "Social welfare & Empowerment", "Utility & Sanitation"], "target_beneficiaries": [{"value": "individual", "label": "Individual"}], "brief_description": "\"GST Exemption Certificate Scheme\" has been introduced by the Dept. of Heavy Industries, for issuing certificates to Persons with Disabilities (PwDs) having a disability percentage equal to or greater than 40%. The scheme provides GSC Concession to orthopedically disabled persons on purchase of car.", "details": "The \"GST Exemption Certificate Scheme\" has been introduced by the Department of Heavy Industries, Ministry of Heavy Industries for issuing certificates to Persons with Disabil

In [48]:
response = llm.invoke(prompt)

print(response.content)

{
    "citizen_questions": [
        "I need financing to purchase a light motor vehicle for my transport business.",
        "Looking for a loan to buy a three‑wheel auto for goods delivery.",
        "Seeking funds to acquire a small van to start a courier service.",
        "Need financial assistance for purchasing a low‑cost vehicle to expand my transport operations.",
        "Looking for a term loan to buy an LMV for my self‑employment venture."
    ]
}


### Generate questions for all schemes
Generate five citizen questions for each selected scheme using category-based few-shot examples. Save each successful result immediately so generation can resume if interrupted.

In [51]:

import json
import time
from pathlib import Path
from tqdm.auto import tqdm

BATCH_SIZE = 10
MAX_RETRIES = 3
RETRY_DELAY = 30

OUTPUT_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "generated_discovery_questions.jsonl"
)

CHECKPOINT_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "generation_checkpoint.json"
)

ERROR_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "generation_errors.jsonl"
)


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return {"completed_scheme_ids": []}


def save_checkpoint(completed_ids):
    temp_path = CHECKPOINT_PATH.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(
            {"completed_scheme_ids": list(completed_ids)},
            f,
            ensure_ascii=False,
            indent=2
        )

    temp_path.replace(CHECKPOINT_PATH)


def generate_one_scheme(scheme):

    examples = select_fewshot_examples(
        scheme,
        fewshot_data,
        max_examples=4
    )

    prompt = build_discovery_prompt(scheme, examples)

    for attempt in range(MAX_RETRIES):

        try:
            response = llm.invoke(prompt)
            content = response.content.strip()

            # Handle Markdown JSON fences
            if content.startswith("```"):
                content = content.split("\n", 1)[1]
                content = content.rsplit("```", 1)[0]

            result = json.loads(content)
            questions = result["citizen_questions"]

            if len(questions) != 5:
                raise ValueError(
                    f"Expected 5 questions, got {len(questions)}"
                )

            return {
                "scheme_id": scheme["scheme_id"],
                "scheme_name": scheme["name"],
                "categories": scheme["categories"],
                "citizen_questions": questions
            }

        except Exception as e:

            if attempt == MAX_RETRIES - 1:
                raise e

            print(
                f"Retry {attempt + 1} for "
                f"{scheme['scheme_id']}: {e}"
            )

            time.sleep(RETRY_DELAY * (attempt + 1))


def generate_in_batches(schemes):

    checkpoint = load_checkpoint()

    completed_ids = set(
        checkpoint["completed_scheme_ids"]
    )

    # Reconcile checkpoint against output file
    if OUTPUT_PATH.exists():
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    completed_ids.add(record["scheme_id"])

    save_checkpoint(completed_ids)

    remaining = [
        scheme for scheme in schemes
        if scheme["scheme_id"] not in completed_ids
    ]

    print("Total schemes:", len(schemes))
    print("Already completed:", len(completed_ids))
    print("Remaining:", len(remaining))

    for batch_start in range(0, len(remaining), BATCH_SIZE):

        batch = remaining[
            batch_start:batch_start + BATCH_SIZE
        ]

        print(
            f"\nBatch {batch_start // BATCH_SIZE + 1} "
            f"| Schemes: {len(batch)}"
        )

        for scheme in tqdm(batch):

            scheme_id = scheme["scheme_id"]

            try:
                record = generate_one_scheme(scheme)

                # Save result immediately
                with open(
                    OUTPUT_PATH, "a", encoding="utf-8"
                ) as f:
                    f.write(
                        json.dumps(record, ensure_ascii=False) + "\n"
                    )

                # Update checkpoint only after successful save
                completed_ids.add(scheme_id)
                save_checkpoint(completed_ids)

            except Exception as e:

                error_record = {
                    "scheme_id": scheme_id,
                    "error": str(e)
                }

                with open(
                    ERROR_PATH, "a", encoding="utf-8"
                ) as f:
                    f.write(
                        json.dumps(error_record, ensure_ascii=False)
                        + "\n"
                    )

                print(f"\nFailed: {scheme_id} | {e}")

                # Stop the run if the API quota/rate limit is exhausted
                error_text = str(e).lower()

                if any(term in error_text for term in [
                    "rate_limit",
                    "rate limit",
                    "resource_exhausted",
                    "quota",
                    "429",
                    "too many requests"
                ]):
                    print("API limit detected. Stopping safely.")
                    return

        print(
            "Completed:",
            len(completed_ids),
            "/",
            len(schemes)
        )

    print("\nGeneration run finished.")
    print("Completed schemes:", len(completed_ids))
    print("Remaining schemes:", len(schemes) - len(completed_ids))


In [52]:
generate_in_batches(discovery_schemes)

Total schemes: 462
Already completed: 7
Remaining: 455

Batch 1 | Schemes: 10


  0%|          | 0/10 [00:00<?, ?it/s]

Completed: 17 / 462

Batch 2 | Schemes: 10


  0%|          | 0/10 [00:00<?, ?it/s]

Completed: 27 / 462

Batch 3 | Schemes: 10


  0%|          | 0/10 [00:00<?, ?it/s]

... (7 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Completed: 77 / 462

Batch 8 | Schemes: 10


  0%|          | 0/10 [00:00<?, ?it/s]

Retry 1 for 8hfXF5gB5uF80_MGxpLX: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199102, Requested 2464. Please try again in 11m16.512s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Retry 2 for 8hfXF5gB5uF80_MGxpLX: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199016, Requested 2464. Please try again in 10m39.36s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Failed: 8hfXF5gB5uF80_MGxpLX | Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-1

In [ ]:
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    generated_data = [
        json.loads(line)
        for line in f if line.strip()
    ]

print("Generated schemes:", len(generated_data))

print(
    "Total questions:",
    sum(len(x["citizen_questions"]) for x in generated_data)
)

print("Failed schemes:", len(generation_errors))